### 1. Pengenalan PEFT (Parameter-Efficient Fine-Tuning)
Saat melatih *Large Language Models* (LLM) ratusan miliar parameter, melakukan *full fine-tuning* (melatih semua bobot) mustahil dilakukan di GPU biasa karena butuh VRAM raksasa. **PEFT** adalah teknik dimana kita **membekukan (freeze) model asli**, dan hanya melatih **sebagian sangat kecil parameter tambahan (adapter)**. Hasilnya: Akurasi bersaing dengan full fine-tuning, namun memori turun drastis!

In [2]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from peft import LoraConfig, get_peft_model, PeftModel, TaskType

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device aktif: {device}")

Device aktif: cuda


### 2. Memuat Model Dasar (Base Model)
Kita muat arsitektur standar tanpa PEFT terlebih dahulu. Model ini berukuran besar dan seluruh sisinya masih berstatus `trainable=True` secara bawaan.

In [3]:
model_id = 'bert-base-uncased'

model = BertForSequenceClassification.from_pretrained(model_id)

def print_params(model :  BertForSequenceClassification):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("Trainable params", trainable_params)
    print("Total params", total_params)

print_params(model)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable params 109483778
Total params 109483778


### 3. Konfigurasi LoRA (Low-Rank Adaptation)
**LoRA** adalah teknik PEFT paling populer dewasa ini. Prinsip kerjanya: menyisipkan matriks berdimensi kecil (*Low-Rank*) ke dalam layer *Attention* LLM. 

Penting dipahami:
- `r`: (Rank) Ukuran dimensi matriks sisipan. Makin kecil (misal 8 atau 16), makin hemat VRAM namun model kurang ekspresif menganalisa pola rumit.
- `lora_alpha`: Faktor pengali (biasanya `r * 2`).
- `target_modules`: Bagian layer mana yang mau disisipkan (di model Transformers biasanya `query`, `value`, atau layer liniar *Dense*).

In [4]:
print(model)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [6]:
loraconfig = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"]
)

### 4. Menerapkan LoRA ke Base Model (`get_peft_model`)
Disinilah letak "sihir" PEFT. Fungsi ini akan secara otomatis membekukan (freeze) semua parameter model inti kita dan menyuntikkan adapter berjejak ringan di target modul kita.

In [7]:
peft_model = get_peft_model(model, loraconfig)
peft_model

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): BertForSequenceClassification(
      (bert): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (token_type_embeddings): Embedding(2, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-11): 12 x BertLayer(
              (attention): BertAttention(
                (self): BertSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): L

In [8]:
print_params(peft_model)

Trainable params 296450
Total params 109780228


In [9]:
peft_model.print_trainable_parameters()

trainable params: 296,450 || all params: 109,780,228 || trainable%: 0.2700


In [16]:
for p in peft_model.bert.base_model.encoder.layer[0].parameters():
    print(p.requires_grad)

False
False
True
True
False
False
False
False
True
True
False
False
False
False
False
False
False
False
False
False


### 5. Simulasi LoRA Training Step
Bagian indahnya dari PEFT adalah: loop training (siklus autograd) yang kita gunakan 100% SAMA dengan PyTorch biasa. PyTorch hanya akan memperbarui bobot LoRA karena bagian lain telah di-freeze (di-set `requires_grad=False`).

In [27]:
peft_model.to(device)
peft_model.train()

tokenizer =  BertTokenizer.from_pretrained(model_id)
inputs = tokenizer('Huawei models are fast!', return_tensors='pt').to(device)
labels = torch.tensor([1]).to(device)

optimizer = torch.optim.AdamW(peft_model.parameters(), lr=1e-3)

peft_model.train()
optimizer.zero_grad()
outputs = peft_model(**inputs, labels=labels)
outputs.loss.backward()
optimizer.step()

### 6. Menyimpan Model PEFT (Hanya Adapternya Saja)
Saat di _save_, PEFT sangat efisien. Alih-alih menyimpan ratusan MB atau Gigabyte parameter *base model*, PEFT hanya menyimpan **file adapter seukuran beberapa MB saja** (`adapter_model.bin` atau folder safetensors)!

In [31]:
save_path = 'lora_bert.pt'

# torch.save(peft_model.state_dict(), save_path)
peft_model.save_pretrained(save_path)

### 7. Memuat & Menggabungkan Model untuk Inferensi (Merge and Unload)
Di kompetisi riil, saat fase test/inference, Anda harus:
1. Memuat ulang model asli (Original Base Model)
2. Memasang/memuat _Adapter_ PEFT yang sudah mengakar di atas base model.
3. Opsional: Melakukan `.merge_and_unload()` untuk meleburkan LoRA tadi menjadi 1 arsitektur base kembali demi kecepatan proses tes.

In [ ]:
original_base = BertForSequenceClassification.from_pretrained(model_id)

inference_model = PeftModel.from_pretrained(original_base, save_path)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


SequenceClassifierOutput(loss=None, logits=tensor([[-0.6676,  0.5890]]), hidden_states=None, attentions=None)


In [40]:
inference_model.eval()
with torch.no_grad():
    inputs = tokenizer('Is solving this competition hard?',return_tensors='pt')
    pred = inference_model(**inputs)
    print(pred.logits.argmax(dim=-1))

tensor([1])


In [42]:
print(inference_model)

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): BertForSequenceClassification(
      (bert): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (token_type_embeddings): Embedding(2, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-11): 12 x BertLayer(
              (attention): BertAttention(
                (self): BertSelfAttention(
                  (query): Linear(in_features=768, out_features=768, bias=True)
                  (key): Linear(in_features=768, out_features=768, bias=True)
                  (value): Linear(in_features=768, out_features=768, bias=True)
                  (dropout): Dropout(p=0.1, inplace=False)
                )
                (outp

In [41]:
merged_model = inference_model.merge_and_unload()
merged_model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [3]:
from torch import nn
class Network(nn.Module):
    def __init__(self):
        super().__init__()
        self.sequential = nn.Sequential(
            nn.Linear(768,768),
            nn.Linear(768,768),
            nn.Linear(768, 5),
            nn.Tanh()
        )
    def forward(self, x):
        x = self.sequential(x)
        return x
network = Network()
network  

Network(
  (sequential): Sequential(
    (0): Linear(in_features=768, out_features=768, bias=True)
    (1): Linear(in_features=768, out_features=768, bias=True)
    (2): Linear(in_features=768, out_features=5, bias=True)
    (3): Tanh()
  )
)

In [4]:
loraconfig = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=['1']
)
peft_model = get_peft_model(network, loraconfig)
peft_model

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): Network(
      (sequential): Sequential(
        (0): Linear(in_features=768, out_features=768, bias=True)
        (1): lora.Linear(
          (base_layer): Linear(in_features=768, out_features=768, bias=True)
          (lora_dropout): ModuleDict(
            (default): Dropout(p=0.1, inplace=False)
          )
          (lora_A): ModuleDict(
            (default): Linear(in_features=768, out_features=8, bias=False)
          )
          (lora_B): ModuleDict(
            (default): Linear(in_features=8, out_features=768, bias=False)
          )
          (lora_embedding_A): ParameterDict()
          (lora_embedding_B): ParameterDict()
          (lora_magnitude_vector): ModuleDict()
        )
        (2): Linear(in_features=768, out_features=5, bias=True)
        (3): Tanh()
      )
    )
  )
)